# Activity: Extend the MCP Server with a van der Waals Tool
In this graded activity, you extend this module's MCP server with a fourth tool, `vanderwaals_pressure`, which computes the pressure of a real gas from the van der Waals equation of state. You implement the handler, write the tool's descriptor and register it, and verify the result at the wire level, entirely in this notebook. There are no `src/` edits: the shipped server code stays identical for all students.

> __Learning Objectives__
>
> By the end of this activity, you will be able to:
> * __Implement a tool handler:__ Write a handler function that validates its arguments, computes the result, and returns a JSON text `String`, throwing an `ArgumentError` on invalid input.
> * __Describe and register a tool:__ Write the tool's JSON Schema descriptor, build the `MCPTool`, and register it alongside the server's default tools.
> * __Verify at the wire level:__ Run real JSON-RPC messages through the server's dispatch loop in-process and check the responses against known values.

Let's get started!
___

## Setup, Data, and Prerequisites
The van der Waals equation of state corrects the ideal gas law for molecular volume and intermolecular attraction:

$$P = \frac{nRT}{V - nb} - \frac{an^{2}}{V^{2}}$$

where $P$ is the pressure (Pa), $n$ is the amount (mol), $T$ is the temperature (K), $V$ is the volume (m³), and $R = 8.314$ J·mol⁻¹·K⁻¹. The species-specific coefficients $a$ (Pa·m⁶·mol⁻²) and $b$ (m³·mol⁻¹) come from the provided table in the module's `data/` directory. The equation requires $V > nb$: the volume must exceed the excluded volume of the molecules. The equation is explicit in $P$, so no root-finding is needed.

First, we set up the computational environment by including the `Include.jl` file, which activates the local environment, loads the required packages, and includes the MCP client and server code in `src/`.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include("Include.jl");

  Activating 

project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-4`


### Constants
We load the van der Waals coefficient table from the module's `data/` directory. Each entry holds the coefficients $a$ and $b$ for one species, keyed by a lowercase species name.

In [2]:
vanderwaals_parameters = JSON.parsefile(joinpath(_PATH_TO_DATA, "vanderwaals-coefficients.json"))

Dict{String, Any} with 6 entries:
  "o2"  => Dict{String, Any}("b"=>3.183e-5, "a"=>0.1378)
  "ch4" => Dict{String, Any}("b"=>4.278e-5, "a"=>0.2283)
  "nh3" => Dict{String, Any}("b"=>3.707e-5, "a"=>0.4225)
  "h2o" => Dict{String, Any}("b"=>3.049e-5, "a"=>0.5536)
  "n2"  => Dict{String, Any}("b"=>3.913e-5, "a"=>0.1408)
  "co2" => Dict{String, Any}("b"=>4.267e-5, "a"=>0.364)

## Task 1: Implement the Handler
Implement the handler for `vanderwaals_pressure`. The contract:

* Takes `arguments::Dict{String,Any}` with keys `species`, `T` (K), `V` (m³), and `n` (mol).
* Returns a JSON text `String` with keys `species`, `P`, and `units`.
* Throws an `ArgumentError` for a missing argument, an unknown species, or `V ≤ nb`.

In [3]:
function vanderwaals_pressure_handler(arguments::Dict{String,Any})::String
    for key in ("species", "T", "V", "n")
        (haskey(arguments, key)) || throw(ArgumentError("Missing required argument: $(key)"));
    end
    species = lowercase(arguments["species"]);
    (haskey(vanderwaals_parameters, species)) || throw(ArgumentError(
        "Unknown species: $(species). Known species: $(join(sort(collect(keys(vanderwaals_parameters))), ", "))"));
    R = 8.314; # J mol⁻¹ K⁻¹
    a = vanderwaals_parameters[species]["a"];
    b = vanderwaals_parameters[species]["b"];
    T = arguments["T"]; V = arguments["V"]; n = arguments["n"];
    (V > n * b) || throw(ArgumentError("V must exceed n*b = $(n * b) m^3"));
    P = n * R * T / (V - n * b) - a * n^2 / V^2;
    return JSON.json(Dict("species" => species, "P" => round(P, sigdigits = 6), "units" => "Pa"));
end

vanderwaals_pressure_handler (generic function with 1 method)

With the handler in place, check it against a hand calculation: CO₂ at $T = 300$ K, $V = 1.0$ L $= 1.0\times10^{-3}$ m³, and $n = 1$ mol gives $P \approx 2.2414\times10^{6}$ Pa (the ideal gas law gives $2.4942\times10^{6}$ Pa at the same state; Task 4 quantifies the difference).

In [4]:
result_check = JSON.parse(vanderwaals_pressure_handler(Dict{String,Any}(
    "species" => "co2", "T" => 300.0, "V" => 1.0e-3, "n" => 1.0)));
@assert isapprox(result_check["P"], 2.24138e6; rtol = 1e-3)
result_check

Dict{String, Any} with 3 entries:
  "units"   => "Pa"
  "P"       => 2.24137e6
  "species" => "co2"

## Task 2: Describe and Register the Tool
A handler alone is not discoverable. Build the `MCPTool` descriptor: the `name` an agent calls, a one-sentence `description`, and an `inputSchema` that requires all four arguments (`species`, `T`, `V`, `n`). Then build the default three-tool registry and register the new tool; afterward, the registry holds four tools.

In [5]:
vanderwaals_tool = build(MCPTool, (
    name = "vanderwaals_pressure",
    description = "Pressure (Pa) of a real gas from the van der Waals equation of state (SI units).",
    inputschema = Dict{String,Any}("type" => "object",
        "properties" => Dict{String,Any}(
            "species" => Dict{String,Any}("type" => "string", "description" => "Species key, e.g. co2, n2, o2, ch4, nh3, h2o"),
            "T" => Dict{String,Any}("type" => "number", "description" => "Temperature in K"),
            "V" => Dict{String,Any}("type" => "number", "description" => "Volume in m^3"),
            "n" => Dict{String,Any}("type" => "number", "description" => "Amount in mol")),
        "required" => ["species", "T", "V", "n"]),
    handler = vanderwaals_pressure_handler));
registry = build_default_registry();
register!(registry, vanderwaals_tool);
@assert length(registry) == 4

## Task 3: Verify at the Wire Level
The registry only matters if it behaves correctly on the wire. The `run_session(...)` function pushes real JSON-RPC lines through the same dispatch loop `server.jl` runs, in-process: message lines go in, response lines come out. The message list below runs a full session: initialize, the initialized notification, `tools/list`, a valid `vanderwaals_pressure` call, and a call whose volume violates $V > nb$. This cell is provided plumbing; the next cell holds the graded checks.

In [6]:
messages = [
    JSON.json(Dict("jsonrpc" => "2.0", "id" => 1, "method" => "initialize", "params" => Dict())),
    JSON.json(Dict("jsonrpc" => "2.0", "method" => "notifications/initialized")),
    JSON.json(Dict("jsonrpc" => "2.0", "id" => 2, "method" => "tools/list")),
    JSON.json(Dict("jsonrpc" => "2.0", "id" => 3, "method" => "tools/call",
        "params" => Dict("name" => "vanderwaals_pressure",
            "arguments" => Dict("species" => "co2", "T" => 300.0, "V" => 1.0e-3, "n" => 1.0)))),
    JSON.json(Dict("jsonrpc" => "2.0", "id" => 4, "method" => "tools/call",
        "params" => Dict("name" => "vanderwaals_pressure",
            "arguments" => Dict("species" => "co2", "T" => 300.0, "V" => 1.0e-5, "n" => 1.0)))),
];
responses = run_session(registry, messages);

In [7]:
response_list = JSON.parse(responses[2]);
toolnames = [t["name"] for t in response_list["result"]["tools"]];
@assert "vanderwaals_pressure" in toolnames
response_call = JSON.parse(responses[3]);
result_wire = JSON.parse(response_call["result"]["content"][1]["text"]);
@assert isapprox(result_wire["P"], 2.24138e6; rtol = 1e-3)
response_domain = JSON.parse(responses[4]);
@assert response_domain["result"]["isError"] == true # V ≤ nb rejected through the wire
println("All wire-level checks passed.")

All wire-level checks passed.


## Task 4: Real-Gas Deviation
How different is the real gas from the ideal one at this state? Call `ideal_gas_solve` through the same registry with the same $T$, $V$, and $n$, and compare its pressure with the van der Waals result from Task 3.

In [8]:
message_ideal = JSON.json(Dict("jsonrpc" => "2.0", "id" => 5, "method" => "tools/call",
    "params" => Dict("name" => "ideal_gas_solve",
        "arguments" => Dict("T" => 300.0, "V" => 1.0e-3, "n" => 1.0))));
response_ideal = JSON.parse(only(run_session(registry, [message_ideal])));
P_ideal = JSON.parse(response_ideal["result"]["content"][1]["text"])["value"];
deviation = (result_wire["P"] - P_ideal) / P_ideal * 100.0;
@assert isapprox(deviation, -10.14; atol = 0.1)
println("van der Waals deviates from ideal by $(round(deviation, digits = 2))% at this state.")

van der Waals deviates from ideal by -10.14% at this state.


## Summary
This activity extended the MCP server with a van der Waals pressure tool and verified it at the wire level, entirely from notebook cells.

> __Key Takeaways:__
>
> * **A handler is validation, computation, and a structured result:** The handler checks its arguments (presence, a known species, and a volume above the molecules' excluded volume), computes the pressure, and returns a JSON text `String`; it throws an `ArgumentError` to signal a tool execution failure.
> * **The descriptor makes the tool discoverable:** The `MCPTool` bundles the name, description, and JSON Schema with the handler, and `register!` adds it to the registry that `tools/list` reports.
> * **Wire-level verification exercises the real dispatch loop:** `run_session` pushes actual JSON-RPC messages through the same dispatch loop the `server.jl` subprocess runs, so checks that pass here describe the tool's behavior on the wire.

The server now exposes your tool exactly as it exposes the shipped three: through a descriptor any MCP client can discover and call.
___